<a href="https://colab.research.google.com/github/arshad831/zain_2026/blob/main/Zain_Customer_360_MCP_LangChain_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zain Jordan Customer 360 MCP + LangChain Demo

## Use Case: MCP-Powered Customer Retention Assistant

This notebook shows a simple enterprise-style MCP workflow:

```text
SQLite Customer 360 Database
        ↓
MCP Server
        ↓
MCP Client
        ↓
LangChain Agent
        ↓
Customer-care recommendation
```

The agent can call MCP tools such as:

- `get_customer_profile`
- `get_customer_churn_risk`
- `get_customer_billing_summary`
- `get_customer_complaints`
- `get_customer_value_segment`

## Assumptions

1. The database is already uploaded to Colab at:

```text
/content/zain_customer_360_ai_demo.db
```

2. Your OpenAI key is saved in Colab Secrets as `OPENAI_API_KEY` or `openai`.

In [1]:
# ============================================================
# 1. Install required packages
# ============================================================

!pip install -q -U fastmcp mcp langchain langchain-openai langchain-community langchain-mcp-adapters nest_asyncio pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.6/738.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.3/216.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142

In [2]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


Libraries imported successfully.


In [14]:
try:
    from google.colab import userdata
    openai_key = userdata.get("openai")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. Agent cells will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


OpenAI API key loaded from Colab Secrets.
OPENAI_API_KEY is available.


In [3]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


Saving zain_customer_360_ai_demo.db to zain_customer_360_ai_demo.db
Uploaded files: ['zain_customer_360_ai_demo.db']


In [4]:
# ============================================================
# 2. Setup OpenAI key and database path
# ============================================================

import os
from pathlib import Path

DB_PATH = "/content/zain_customer_360_ai_demo.db"
MODEL_NAME = "gpt-4.1-mini"

try:
    from google.colab import userdata

    api_key = userdata.get("OPENAI_API_KEY") or userdata.get("openai")

    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key
        print("OPENAI_API_KEY loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")

except Exception:
    print("Not running in Colab, or Colab Secrets not available.")

if Path(DB_PATH).exists():
    print("Database found:", DB_PATH)
else:
    print("Database not found. Please upload zain_customer_360_ai_demo.db to /content/")

Not running in Colab, or Colab Secrets not available.
Database found: /content/zain_customer_360_ai_demo.db


In [5]:
# ============================================================
# 3. Quick database inspection
# ============================================================

import sqlite3
import pandas as pd

if Path(DB_PATH).exists():
    conn = sqlite3.connect(DB_PATH)

    tables_df = pd.read_sql_query(
        '''
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        ''',
        conn
    )

    display(tables_df)

    conn.close()
else:
    print("Database not found. Upload the database first.")

,name
0,accounts
1,addons
2,call_detail_records
3,campaigns
4,complaints
5,customer_campaign_responses
6,customer_churn_scores
7,customer_monthly_summary
8,customer_satisfaction
9,customer_value_segments


In [18]:
# ============================================================
# 4. Create the MCP server file
# ============================================================

%%writefile zain_customer_mcp_server.py
import os
import json
import sqlite3
from pathlib import Path
from typing import List

from fastmcp import FastMCP


mcp = FastMCP("zain_customer_360_mcp_server")

DB_PATH = os.environ.get("ZAIN_DB_PATH", "/content/zain_customer_360_ai_demo.db")


def rows_to_json(rows: List[sqlite3.Row]) -> str:
    data = [dict(row) for row in rows]
    return json.dumps(data, indent=2, ensure_ascii=False, default=str)


def run_select(query: str, params: tuple = ()) -> str:
    if not Path(DB_PATH).exists():
        return json.dumps({
            "error": f"Database not found at {DB_PATH}",
            "fix": "Upload zain_customer_360_ai_demo.db to /content/"
        }, indent=2)

    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(query, params)
        rows = cursor.fetchall()
        conn.close()

        if not rows:
            return json.dumps({
                "found": False,
                "message": "No records found.",
                "params": params
            }, indent=2)

        return rows_to_json(rows)

    except Exception as e:
        return json.dumps({
            "error": str(e),
            "query_used": query,
            "params": params
        }, indent=2)


def table_has_column(table_name: str, column_name: str) -> bool:
    try:
        conn = sqlite3.connect(DB_PATH)
        cols = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
        conn.close()
        return any(col[1] == column_name for col in cols)
    except Exception:
        return False


@mcp.tool()
def get_customer_profile(customer_id: int) -> str:
    query = '''
    SELECT *
    FROM customers
    WHERE customer_id = ?
    LIMIT 1;
    '''
    return run_select(query, (customer_id,))


@mcp.tool()
def get_customer_churn_risk(customer_id: int) -> str:
    query = '''
    SELECT *
    FROM customer_churn_scores
    WHERE customer_id = ?
    LIMIT 1;
    '''
    return run_select(query, (customer_id,))


@mcp.tool()
def get_customer_value_segment(customer_id: int) -> str:
    query = '''
    SELECT *
    FROM customer_value_segments
    WHERE customer_id = ?
    LIMIT 1;
    '''
    return run_select(query, (customer_id,))


@mcp.tool()
def get_customer_complaints(customer_id: int, limit: int = 5) -> str:
    limit = max(1, min(int(limit), 10))

    query = f'''
    SELECT *
    FROM complaints
    WHERE customer_id = ?
    LIMIT {limit};
    '''
    return run_select(query, (customer_id,))


@mcp.tool()
def get_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    limit = max(1, min(int(limit), 10))

    if table_has_column("invoices", "customer_id"):
        query = f'''
        SELECT *
        FROM invoices
        WHERE customer_id = ?
        LIMIT {limit};
        '''
        return run_select(query, (customer_id,))

    query = f'''
    SELECT i.*
    FROM invoices i
    JOIN accounts a
        ON i.account_id = a.account_id
    WHERE a.customer_id = ?
    LIMIT {limit};
    '''
    return run_select(query, (customer_id,))


@mcp.tool()
def get_customer_plan(customer_id: int) -> str:
    query = '''
    SELECT
        s.*,
        p.*
    FROM subscriptions s
    LEFT JOIN plans p
        ON s.plan_id = p.plan_id
    WHERE s.customer_id = ?
    LIMIT 5;
    '''
    return run_select(query, (customer_id,))


if __name__ == "__main__":
    mcp.run(
        transport="http",
        host="127.0.0.1",
        port=8000,
        path="/mcp"
    )

Overwriting zain_customer_mcp_server.py


In [19]:
# ============================================================
# 5. Syntax check the MCP server file
# ============================================================

!python -m py_compile zain_customer_mcp_server.py

print("MCP server file syntax is OK.")

MCP server file syntax is OK.


In [20]:
# ============================================================
# 6. Start the MCP server in the background
# ============================================================

import subprocess
import time
import socket

MCP_HOST = "127.0.0.1"
MCP_PORT = 8000
MCP_SERVER_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

os.environ["ZAIN_DB_PATH"] = DB_PATH


def wait_for_port(host: str, port: int, timeout: int = 25) -> bool:
    start = time.time()

    while time.time() - start < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)

    return False


try:
    if server_process.poll() is None:
        server_process.terminate()
        time.sleep(2)
        print("Old MCP server stopped.")
except NameError:
    pass


log_file = open("mcp_server.log", "w")

server_process = subprocess.Popen(
    ["python", "-u", "zain_customer_mcp_server.py"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

if wait_for_port(MCP_HOST, MCP_PORT, timeout=25):
    print("MCP server is running:")
    print(MCP_SERVER_URL)
else:
    print("MCP server did not start. Showing log:")
    log_file.flush()
    !cat mcp_server.log
    raise RuntimeError("MCP server failed to start.")

Old MCP server stopped.
MCP server is running:
http://127.0.0.1:8000/mcp


In [21]:
# ============================================================
# 7. Load MCP tools into LangChain
# ============================================================

import nest_asyncio
nest_asyncio.apply()

from langchain_mcp_adapters.client import MultiServerMCPClient


async def load_mcp_tools():
    errors = []

    for transport_name in ["streamable_http", "http"]:
        try:
            client = MultiServerMCPClient(
                {
                    "zain_customer_360": {
                        "transport": transport_name,
                        "url": MCP_SERVER_URL,
                    }
                }
            )

            tools = await client.get_tools()
            return client, tools, transport_name

        except Exception as e:
            errors.append(f"{transport_name}: {repr(e)}")

    raise RuntimeError("Could not load MCP tools:\\n" + "\\n".join(errors))


mcp_client, mcp_tools, used_transport = await load_mcp_tools()

print(f"MCP tools loaded using transport='{used_transport}'")
print(f"Number of tools: {len(mcp_tools)}")

for tool in mcp_tools:
    print("-", tool.name, ":", (tool.description or "")[:120])

MCP tools loaded using transport='streamable_http'
Number of tools: 6
- get_customer_profile : 
- get_customer_churn_risk : 
- get_customer_value_segment : 
- get_customer_complaints : 
- get_customer_billing_summary : 
- get_customer_plan : 


In [22]:
# ============================================================
# 8. Test one MCP tool directly
# ============================================================

def find_tool(tool_name: str):
    for tool in mcp_tools:
        if tool.name == tool_name or tool.name.endswith(tool_name):
            return tool
    raise ValueError(f"Tool not found: {tool_name}")


profile_tool = find_tool("get_customer_profile")
profile_result = await profile_tool.ainvoke({"customer_id": 42})

print(profile_result)

[{'type': 'text', 'text': '[\n  {\n    "customer_id": 42,\n    "nationality": "Jordanian",\n    "full_name": "Ali Al-Salem",\n    "gender": "Male",\n    "date_of_birth": "1982-06-18",\n    "age_group": "35-44",\n    "city": "Amman",\n    "governorate": "Amman",\n    "customer_type": "Individual",\n    "customer_segment": "Family",\n    "preferred_language": "Arabic",\n    "email": "ali.42@example.com",\n    "phone_number": "+962799389915",\n    "signup_date": "2023-02-06",\n    "status": "Active"\n  }\n]', 'id': 'lc_bf1f93c7-0f51-4c2b-9264-df6080c591e3'}]


In [23]:
# ============================================================
# 9. Test churn and value tools directly
# ============================================================

churn_tool = find_tool("get_customer_churn_risk")
value_tool = find_tool("get_customer_value_segment")

print("CHURN RISK")
print(await churn_tool.ainvoke({"customer_id": 42}))

print("\nVALUE SEGMENT")
print(await value_tool.ainvoke({"customer_id": 42}))

CHURN RISK
[{'type': 'text', 'text': '[\n  {\n    "churn_score_id": 42,\n    "customer_id": 42,\n    "score_month": "2026-04",\n    "churn_score": 0.349,\n    "risk_level": "Medium",\n    "main_risk_reason": "Competitor risk",\n    "recommended_action": "Invite to loyalty campaign"\n  }\n]', 'id': 'lc_c1d0ab8c-860c-4694-94a9-8bb90897a4e1'}]

VALUE SEGMENT
[{'type': 'text', 'text': '[\n  {\n    "value_segment_id": 42,\n    "customer_id": 42,\n    "segment_month": "2026-04",\n    "arpu_jod": 14.146,\n    "total_revenue_6m_jod": 84.875,\n    "value_segment": "Low Value",\n    "lifetime_months": 39\n  }\n]', 'id': 'lc_3bb71614-fd47-41ca-a544-c6d35ab1d5ab'}]


In [24]:
# ============================================================
# 10. Create LangChain agent using MCP tools
# ============================================================

from langchain.agents import create_agent

mcp_agent = create_agent(
    model=f"openai:{MODEL_NAME}",
    tools=mcp_tools,
    system_prompt=(
        "You are a professional Zain Jordan customer-care and retention assistant. "
        "Your tools are provided through an MCP server. "
        "Use MCP tools whenever customer data is needed. "
        "Do not invent customer facts. "
        "If data is missing, say what is missing. "
        "Keep the answer structured and business-friendly. "
        "Final answer sections: Customer Summary, Plan Summary, Churn and Value Risk, "
        "Evidence from Data, Recommended Next Action, Customer-Care Message."
    ),
)

print("LangChain agent with MCP tools is ready.")

LangChain agent with MCP tools is ready.


In [25]:
# ============================================================
# 11. Run the main MCP customer retention demo
# ============================================================

demo_prompt = '''
Use the Zain Jordan MCP tools to analyze customer 42.

Check:
1. customer profile
2. current plan
3. churn risk
4. billing summary
5. recent complaints
6. customer value segment

Then provide:
1. Customer Summary
2. Plan Summary
3. Churn and Value Risk
4. Evidence from Data
5. Recommended Next Action
6. Short Customer-Care Message
'''

response = await mcp_agent.ainvoke(
    {"messages": [{"role": "user", "content": demo_prompt}]}
)

print(response["messages"][-1].content)

Customer Summary:
- Customer ID: 42
- Name: Ali Al-Salem
- Nationality: Jordanian
- Age Group: 35-44
- Location: Amman, Amman Governorate
- Customer Type: Individual, Family Segment
- Preferred Language: Arabic
- Status: Active since 2023-02-06

Plan Summary:
- Plan Name: Shabab 5 (Mobile Prepaid)
- Monthly Fee: 5.0 JOD
- Data Allowance: 6 GB
- Local Minutes: 250
- SMS Allowance: 50
- Technology: 4G
- Activation Date: 2025-07-24
- Auto-renewal: Disabled

Churn and Value Risk:
- Churn Risk Level: Medium
- Main Risk Reason: Competitor risk
- Recommended Action: Invite to loyalty campaign
- Value Segment: Low Value
- Average Revenue Per User (ARPU): 14.146 JOD
- Total Revenue in Last 6 Months: 84.875 JOD
- Customer Lifetime: 39 months

Evidence from Data:
- Customer has timely bill payments with no overdue days.
- No complaints recorded recently.
- Medium churn risk due to competitor offerings.
- Customer is categorized in a low-value segment with modest ARPU.

Recommended Next Action:
- 

In [17]:
# ============================================================
# 12. Optional exercise: compare two customers
# ============================================================

compare_prompt = '''
Use the Zain Jordan MCP tools to compare customer 25 and customer 42.

For each customer, check:
- profile
- churn risk
- value segment
- recent complaints
- billing summary

Then tell the retention team:
1. Who should be prioritized first?
2. Why?
3. What action should be taken for each customer?
'''

response = await mcp_agent.ainvoke(
    {"messages": [{"role": "user", "content": compare_prompt}]}
)

print(response["messages"][-1].content)

Customer 25 and Customer 42 have been compared on the following parameters:

Customer Summary:
- Customer 25: Mahmoud Al-Ahmad, Male, 18-24 age group, SME segment, living in Aqaba. Active since 2019.
- Customer 42: Ali Al-Salem, Male, 35-44 age group, Family segment, living in Amman. Active since 2023.

Churn Risk:
- Customer 25 has a medium churn risk with a score of 0.305 mainly due to low engagement. Recommended action is offering data bundles.
- Customer 42 also has a medium churn risk but slightly higher at 0.349, mainly due to competitor risk. Recommended action is inviting to loyalty campaign.

Value Segment:
- Customer 25 is a VIP segment with a high ARPU of 151.949 JOD and total revenue over 6 months of 911.695 JOD.
- Customer 42 is in the Low Value segment with a low ARPU of 14.146 JOD and total revenue over 6 months of 84.875 JOD.

Recent Complaints:
- Neither customer has any recent complaints on record.

Billing Summary:
- Customer 25 has consistently paid higher monthly b

# Teaching Script

Use this explanation during the live class:

> In Class 2, our tools were Python functions inside the notebook.  
> In Class 6, we expose similar customer capabilities through an MCP server.  
> The LangChain agent no longer directly owns the database functions.  
> It loads approved tools from the MCP server and calls them when needed.

## Why this matters

MCP makes the architecture more enterprise-ready because:

- tools are exposed through a standard interface,
- different agents can reuse the same tools,
- the agent does not need unrestricted database access,
- tool boundaries are clearer,
- the same customer capabilities can support capstone projects.

## Best demo line

> MCP turns local notebook functions into reusable enterprise AI tools.